# MevzuatRadar — mT5 ile değişiklik çıkarımı

Bu defter, kural sistemi + doğrulama ile üretilen gümüş etiketlerle bir **mT5-small** modelini eğitir ve
geliştirme setinde tahmin üretir. Tahminler daha sonra bilgisayarda `mevzuatradar evaluate-model` ile
kural sistemiyle aynı doğrulamadan geçirilir.

**Başlamadan önce:** Üst menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçin.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.46" datasets sentencepiece accelerate

## 1. Veriyi yükle
Bilgisayarındaki `data/ml/seq2seq_train.jsonl` ve `data/ml/seq2seq_dev.jsonl` dosyalarını seç.

In [ ]:
from google.colab import files
yuklenen = files.upload()
print(list(yuklenen))

In [ ]:
import json, random
import numpy as np
import torch

def oku(yol):
    return [json.loads(l) for l in open(yol, encoding="utf-8")]

train = oku("seq2seq_train.jsonl")
dev = oku("seq2seq_dev.jsonl")
GUVENILIR = {"dogrulanmis", "kayitsiz"}
dev_guvenilir = [r for r in dev if r["quality"] in GUVENILIR]   # model seçimi yalnızca bunlarla
print(f"eğitim {len(train)} | geliştirme {len(dev)} (güvenilir {len(dev_guvenilir)})")
print("\nÖrnek girdi :", train[0]["input"][:300])
print("Örnek hedef :", train[0]["target"])

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Model ve tokenizer
`google/mt5-small`: Türkçe dahil 101 dilde ön eğitilmiş, ~300M parametreli bir encoder-decoder model.
Uzunluk sınırları, `export-seq2seq` çıktısındaki en uzun girdi/hedef değerlerine göre seçildi.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "google/mt5-small"
ONEK = "degisiklik: "
MAX_GIRDI, MAX_HEDEF = 384, 256

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

uzunluk = lambda xs: max(len(tokenizer(x).input_ids) for x in xs)
print("en uzun girdi (token):", uzunluk([ONEK + r["input"] for r in train + dev]))
print("en uzun hedef (token):", uzunluk([r["target"] for r in train + dev]))

In [ ]:
from datasets import Dataset

def hazirla(orn):
    enc = tokenizer([ONEK + x for x in orn["input"]], max_length=MAX_GIRDI, truncation=True)
    enc["labels"] = tokenizer(text_target=orn["target"], max_length=MAX_HEDEF, truncation=True)["input_ids"]
    return enc

sutunlar = ["input", "target"]
ds_train = Dataset.from_list([{k: r[k] for k in sutunlar} for r in train]).map(hazirla, batched=True, remove_columns=sutunlar)
ds_dev = Dataset.from_list([{k: r[k] for k in sutunlar} for r in dev_guvenilir]).map(hazirla, batched=True, remove_columns=sutunlar)

## 3. Eğitim
Veri küçük olduğu için çok sayıda epoch ve T5 ailesinde yaygın kullanılan Adafactor iyileştirici kullanılıyor.
Her epoch sonunda güvenilir geliştirme örneklerinde **tam eşleşme** ölçülür ve en iyi model saklanır.
(mT5, fp16 ile kararsız olabildiği için tam hassasiyette eğitiliyor.)

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

def normalize(s):
    parcalar = [p.strip() for p in s.split(";;") if p.strip()]
    return " ;; ".join(sorted(parcalar))

def metrikler(tahmin):
    preds, labels = tahmin
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    p = tokenizer.batch_decode(preds, skip_special_tokens=True)
    g = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {"tam_eslesme": float(np.mean([normalize(a) == normalize(b) for a, b in zip(p, g)]))}

args = Seq2SeqTrainingArguments(
    output_dir="mt5_mevzuat",
    num_train_epochs=40,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    optim="adafactor",
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="tam_eslesme",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_HEDEF,
    generation_num_beams=1,
    logging_steps=10,
    report_to="none",
    seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=ds_train, eval_dataset=ds_dev,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer, compute_metrics=metrikler,
)
trainer.train()
print("En iyi tam eşleşme:", trainer.state.best_metric)

## 4. Geliştirme setinin TAMAMINDA tahmin
Model, kural sistemi gibi her maddede tahmin yapar (yalnızca güvenilir olanlarda değil).

In [ ]:
model = trainer.model.eval()
cihaz = model.device
tahminler = []
for i in range(0, len(dev), 8):
    grup = dev[i:i + 8]
    enc = tokenizer([ONEK + r["input"] for r in grup], max_length=MAX_GIRDI, truncation=True,
                    padding=True, return_tensors="pt").to(cihaz)
    with torch.no_grad():
        cikti = model.generate(**enc, max_new_tokens=MAX_HEDEF, num_beams=4)
    for r, t in zip(grup, tokenizer.batch_decode(cikti, skip_special_tokens=True)):
        tahminler.append({"id": r["id"], "prediction": t})

with open("preds_dev.jsonl", "w", encoding="utf-8") as f:
    for p in tahminler:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

for r, p in list(zip(dev, tahminler))[:5]:
    print("\nGİRDİ :", r["input"][:200])
    print("HEDEF :", r["target"])
    print("MODEL :", p["prediction"])

In [ ]:
files.download("preds_dev.jsonl")

## 5. Sonra
İndirilen `preds_dev.jsonl` dosyasını projede `data/ml/` klasörüne koyup bilgisayarda çalıştır:

```
mevzuatradar evaluate-model --pred data/ml/preds_dev.jsonl --split dev
```